# License Plate Detection — Model 2: Faster R-CNN
**CMPS 261 — Machine Learning Project**

Faster R-CNN is a two-stage detector:
1. **Region Proposal Network (RPN)** — proposes candidate regions that may contain an object
2. **Detection head** — classifies and refines each proposed region

It is generally more accurate than YOLO on small datasets but slower at inference.

## 1. Setup

In [ ]:
import sys, os, json, random, time
sys.path.append('..')

import torch
import torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2, FasterRCNN_ResNet50_FPN_V2_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torch.utils.data import DataLoader, random_split
import torchvision.transforms as T
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import numpy as np
from tqdm import tqdm

from src.fasterrcnn_dataset import LicensePlateDataset, collate_fn

DEVICE = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

# Paths
IMG_DIR = '../data/archive/images'
ANN_DIR = '../data/archive/annotations'
MODEL_SAVE_PATH = '../models/fasterrcnn_best.pth'

## 2. Dataset & Splits

In [ ]:
SEED = 42
torch.manual_seed(SEED)
random.seed(SEED)

train_transforms = T.Compose([
    T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    T.ToTensor(),
])
val_transforms = T.Compose([T.ToTensor()])

full_dataset = LicensePlateDataset(IMG_DIR, ANN_DIR)
n = len(full_dataset)
n_train = int(0.70 * n)
n_val   = int(0.15 * n)
n_test  = n - n_train - n_val

train_ds, val_ds, test_ds = random_split(
    full_dataset, [n_train, n_val, n_test],
    generator=torch.Generator().manual_seed(SEED)
)

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True,  collate_fn=collate_fn, num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=4, shuffle=False, collate_fn=collate_fn, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=4, shuffle=False, collate_fn=collate_fn, num_workers=0)

print(f'Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')

## 3. Build the Model
We use a pretrained ResNet-50 FPN backbone and replace the box predictor head for our 1-class problem.

In [ ]:
def build_model(num_classes=2):  # 1 class + background
    weights = FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT
    model   = fasterrcnn_resnet50_fpn_v2(weights=weights)
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    return model

model = build_model(num_classes=2)
model.to(DEVICE)
print('Model ready.')

params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

## 4. Training Loop

In [ ]:
def train_one_epoch(model, optimizer, loader, device):
    model.train()
    total_loss = 0
    for images, targets in tqdm(loader, desc='Train', leave=False):
        images  = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        loss_dict = model(images, targets)
        loss = sum(loss_dict.values())
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def evaluate_loss(model, loader, device):
    model.train()  # keep train mode to get loss dict
    total_loss = 0
    for images, targets in tqdm(loader, desc='Val  ', leave=False):
        images  = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        loss_dict = model(images, targets)
        total_loss += sum(loss_dict.values()).item()
    return total_loss / len(loader)


NUM_EPOCHS  = 30
best_val    = float('inf')
train_losses, val_losses = [], []

os.makedirs('../models', exist_ok=True)

for epoch in range(1, NUM_EPOCHS + 1):
    t0         = time.time()
    train_loss = train_one_epoch(model, optimizer, train_loader, DEVICE)
    val_loss   = evaluate_loss(model, val_loader, DEVICE)
    scheduler.step()

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    if val_loss < best_val:
        best_val = val_loss
        torch.save(model.state_dict(), MODEL_SAVE_PATH)
        flag = ' ← best'
    else:
        flag = ''

    print(f'Epoch {epoch:2d}/{NUM_EPOCHS} | '
          f'Train loss: {train_loss:.4f} | '
          f'Val loss: {val_loss:.4f} | '
          f'{time.time()-t0:.0f}s{flag}')

## 5. Loss Curve

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses,   label='Val Loss')
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.title('Faster R-CNN — Training & Validation Loss')
plt.legend(); plt.tight_layout()
plt.savefig('../results/fasterrcnn_loss_curve.png', dpi=150)
plt.show()
print('Saved: results/fasterrcnn_loss_curve.png')

## 6. IoU Evaluation on Test Set
We compute mean IoU (Intersection over Union) across all test images.

In [ ]:
def compute_iou(boxA, boxB):
    """Compute IoU between two boxes [x1,y1,x2,y2]."""
    xA = max(boxA[0], boxB[0]); yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2]); yB = min(boxA[3], boxB[3])
    inter = max(0, xB - xA) * max(0, yB - yA)
    areaA = (boxA[2]-boxA[0]) * (boxA[3]-boxA[1])
    areaB = (boxB[2]-boxB[0]) * (boxB[3]-boxB[1])
    return inter / (areaA + areaB - inter + 1e-6)


# Load best model
model.load_state_dict(torch.load(MODEL_SAVE_PATH, map_location=DEVICE))
model.eval()

iou_scores = []
tp50 = fp50 = fn50 = 0   # for mAP@0.5
CONF_THRESH = 0.5
IOU_THRESH  = 0.5

with torch.no_grad():
    for images, targets in tqdm(test_loader, desc='Evaluating'):
        images = [img.to(DEVICE) for img in images]
        preds  = model(images)

        for pred, target in zip(preds, targets):
            gt_boxes   = target['boxes'].numpy()
            pred_boxes = pred['boxes'][pred['scores'] >= CONF_THRESH].cpu().numpy()
            pred_scores= pred['scores'][pred['scores'] >= CONF_THRESH].cpu().numpy()

            matched_gt = set()
            for pb in pred_boxes:
                best_iou, best_j = 0, -1
                for j, gb in enumerate(gt_boxes):
                    iou = compute_iou(pb, gb)
                    if iou > best_iou:
                        best_iou, best_j = iou, j
                if best_iou >= IOU_THRESH and best_j not in matched_gt:
                    tp50 += 1
                    matched_gt.add(best_j)
                    iou_scores.append(best_iou)
                else:
                    fp50 += 1
            fn50 += len(gt_boxes) - len(matched_gt)

precision = tp50 / (tp50 + fp50 + 1e-6)
recall    = tp50 / (tp50 + fn50 + 1e-6)
f1        = 2 * precision * recall / (precision + recall + 1e-6)
mean_iou  = float(np.mean(iou_scores)) if iou_scores else 0.0

print(f'Test Results (conf>={CONF_THRESH}, IoU>={IOU_THRESH})')
print(f'  Precision : {precision:.4f}')
print(f'  Recall    : {recall:.4f}')
print(f'  F1        : {f1:.4f}')
print(f'  Mean IoU  : {mean_iou:.4f}')

## 7. Visualise Predictions

In [ ]:
model.eval()
sample_indices = random.sample(range(len(test_ds)), 8)

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

with torch.no_grad():
    for ax, idx in zip(axes, sample_indices):
        img_tensor, target = test_ds[idx]
        pred = model([img_tensor.to(DEVICE)])[0]

        img_np = img_tensor.permute(1, 2, 0).numpy()
        ax.imshow(img_np)

        # Ground truth (red)
        for box in target['boxes']:
            x1,y1,x2,y2 = box.tolist()
            ax.add_patch(patches.Rectangle((x1,y1), x2-x1, y2-y1,
                         linewidth=2, edgecolor='red', facecolor='none', label='GT'))

        # Predictions (lime)
        for box, score in zip(pred['boxes'], pred['scores']):
            if score < 0.5: continue
            x1,y1,x2,y2 = box.cpu().tolist()
            ax.add_patch(patches.Rectangle((x1,y1), x2-x1, y2-y1,
                         linewidth=2, edgecolor='lime', facecolor='none'))
            ax.text(x1, y1-4, f'{score:.2f}', color='lime', fontsize=8,
                    bbox=dict(facecolor='black', alpha=0.4, pad=1))

        ax.axis('off')

plt.suptitle('Faster R-CNN — Test Predictions (red=GT, lime=pred)', fontsize=12)
plt.tight_layout()
plt.savefig('../results/fasterrcnn_predictions.png', dpi=150)
plt.show()
print('Saved: results/fasterrcnn_predictions.png')

## 8. Save Metrics for Comparison

In [ ]:
frcnn_metrics = {
    'model'    : 'Faster R-CNN (ResNet50-FPN)',
    'precision': round(precision, 4),
    'recall'   : round(recall, 4),
    'f1'       : round(f1, 4),
    'mean_iou' : round(mean_iou, 4),
}

with open('../results/fasterrcnn_metrics.json', 'w') as f:
    json.dump(frcnn_metrics, f, indent=2)

print('Metrics saved to results/fasterrcnn_metrics.json')
print(json.dumps(frcnn_metrics, indent=2))